# ORNJ Phase 1–3: Real-Data Clinical/Dosimetric Models + Public CT Imaging Workflow

**Authors:** Seher Siddiqui and Arnav Jain · B.Tech AI · NMIMS MPSTME

## Data integrity: real data 

This notebook uses two independent real datasets:
1. **ORN outcomes cohort** — the supplied MD Anderson / Figshare table (1,129 patients, 198 ORN events): clinical covariates, dose-volume histogram (DVH) summaries, event times, survival and ORN grade.
2. **Public head-and-neck CT cohort** — TCIA HNSCC-3DCT-RT, a real longitudinal CT + RTSTRUCT/RTDOSE dataset (31 participants; pre-, mid- and post-treatment CT). The full imaging archive is about 14 GB and may require restricted-access agreement / TCIA Data Retriever. See [TCIA collection page](https://www.cancerimagingarchive.net/collection/hnscc-3dct-rt/) and dataset DOI [10.7937/K9/TCIA.2018.13upr2xf](https://doi.org/10.7937/K9/TCIA.2018.13upr2xf).

## 0. Setup and reproducible data loading

In [1]:
import os, sys, warnings, math, random, gzip, base64, io, glob
import numpy as np, pandas as pd
import plotly.express as px, plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score, brier_score_loss,
    roc_curve, precision_recall_curve, confusion_matrix, balanced_accuracy_score, precision_score, recall_score)
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
warnings.filterwarnings("ignore")
SEED=42
np.random.seed(SEED); random.seed(SEED)
DATA_B64_GZIP='...'
raw=pd.read_csv(io.BytesIO(gzip.decompress(base64.b64decode(DATA_B64_GZIP))))
print(f"Loaded real ORN cohort: {raw.shape[0]:,} patients × {raw.shape[1]} columns")
print("Missing cells:", int(raw.isna().sum().sum()), "| duplicate IDs:", int(raw["Study ID"].duplicated().sum()))
display(raw.head())


Loaded real ORN cohort: 1,129 patients × 61 columns
Missing cells: 0 | duplicate IDs: 0


,Study ID,Gender,Age,Overall survival,Overall survival time,ORN status,Time to event,ORN Grade (Tsai),Smoking status,Smoking Pack-Years,Pre-RT dental extractions,T Stage,N Stage,Chemotherapy,Postop RT vs. Definitive RT,HPV/P16 +Ve,Tumor site group,Mandible volume (cc),V5Gy,V10Gy,V15Gy,V20Gy,V25Gy,V30Gy,V35Gy,V40Gy,V45Gy,V50Gy,V55Gy,V60Gy,V65Gy,V70Gy,V75Gy,V80Gy,D0.5%,D1%,D2%,D3%,D5%,D10%,D15%,D20%,D25%,D30%,D35%,D40%,D45%,D50%,D55%,D60%,D65%,D70%,D75%,D80%,D85%,D90%,D95%,D97%,D98%,D99%,D99.5%
0,2,Male,58,0,117.0,0,117.40,0,Never,0.0,0,T4,N0,No Chemotherapy,postop_RT,Unknown,Oral Cavity,113.43,100.00,90.98,81.84,67.84,62.39,54.82,47.61,41.89,38.04,35.21,31.04,5.31,0.00,0.0,0.0,0.0,62.35,62.07,61.63,61.18,60.13,59.04,58.66,58.27,57.80,55.91,50.24,41.92,36.84,33.13,29.90,27.06,22.44,18.73,17.27,15.90,12.85,10.42,8.29,7.52,7.02,6.53,6.30
1,4,Male,58,1,45.6,0,45.40,0,Never,0.0,1,T2,N0,Concurrent Chemotherapy,postop_RT,Unknown,Oral Cavity,84.56,98.34,92.20,89.98,87.53,86.29,84.76,83.01,79.73,70.22,55.45,31.49,5.86,0.00,0.0,0.0,0.0,62.74,61.96,61.37,60.89,60.20,59.01,57.97,57.09,56.14,55.22,54.52,53.62,52.52,51.18,50.10,48.65,47.08,45.09,42.90,39.74,29.65,14.91,7.18,5.93,5.23,3.72,2.59
2,8,Male,82,1,134.2,0,117.50,0,Former,70.0,0,T2,N1,No Chemotherapy,postop_RT,Unknown,Oral Cavity,88.24,99.79,97.77,93.73,92.05,90.33,86.89,83.80,78.34,70.26,56.73,35.93,7.95,0.00,0.0,0.0,0.0,61.53,61.26,60.96,60.79,60.47,59.66,58.72,57.54,56.78,55.97,55.20,54.14,52.83,51.76,50.45,49.09,47.27,45.12,42.07,39.17,33.16,25.49,12.51,10.62,9.80,8.18,6.42
3,9,Male,70,1,46.2,0,46.00,0,Former,35.0,0,T1,N2A,No Chemotherapy,Definitive,Unknown,Oropharynx,62.10,97.51,90.61,85.16,82.55,78.90,66.20,50.49,36.75,30.20,27.19,23.30,15.59,4.58,0.0,0.0,0.0,66.88,66.56,66.17,65.75,64.78,62.69,60.40,57.03,53.48,45.19,41.00,38.20,36.36,35.12,33.98,32.34,30.28,28.64,26.87,23.97,15.33,10.40,7.00,5.57,4.60,3.67,3.03
4,15,Female,66,0,86.0,1,5.45,4,Never,0.0,1,T1,N0,Concurrent Chemotherapy,postop_RT,Unknown,Oral Cavity,65.90,83.99,80.40,77.94,76.89,75.73,74.77,73.61,72.65,70.81,68.69,60.77,34.69,1.02,0.0,0.0,0.0,66.44,65.04,63.75,63.35,62.91,62.26,61.71,61.33,60.96,60.53,59.97,59.25,58.30,57.56,56.76,55.22,53.48,48.14,28.28,10.60,4.31,2.01,1.19,0.98,0.88,0.81,0.78


## 1. Phase 1 — cohort audit and exploratory analysis

In [2]:
target='ORN status'
y=raw[target].astype(int)
grade_counts=raw['ORN Grade (Tsai)'].value_counts().sort_index()
audit=pd.DataFrame({'dtype':raw.dtypes.astype(str),'missing_n':raw.isna().sum(),'unique_n':raw.nunique()})
display(audit)
print(f"ORN incidence: {y.sum()}/{len(y)} = {y.mean():.1%}")
print("Grade counts:", grade_counts.to_dict())
fig=make_subplots(rows=1,cols=3,specs=[[{'type':'domain'},{'type':'xy'},{'type':'xy'}]],
                  subplot_titles=['ORN outcome','Tsai grade','Age by outcome'])
fig.add_trace(go.Pie(labels=['No ORN','ORN'],values=[(y==0).sum(),(y==1).sum()],hole=.45),1,1)
fig.add_trace(go.Bar(x=grade_counts.index.astype(str),y=grade_counts.values,text=grade_counts.values,textposition='auto'),1,2)
for k,label in [(0,'No ORN'),(1,'ORN')]:
    fig.add_trace(go.Histogram(x=raw.loc[y==k,'Age'],name=label,opacity=.65),1,3)
fig.update_layout(height=420,barmode='overlay',title='Real ORN cohort overview'); fig.show()
site=raw.groupby('Tumor site group')['ORN status'].agg(['count','sum','mean']).reset_index()
site['prevalence_pct']=100*site['mean']
display(site.sort_values('prevalence_pct',ascending=False))
px.bar(site,x='Tumor site group',y='prevalence_pct',hover_data=['count','sum'],
       title='Observed ORN prevalence by tumour site').update_layout(xaxis_tickangle=-25).show()


,dtype,missing_n,unique_n
Study ID,int64,0,1129
Gender,object,0,2
Age,int64,0,62
Overall survival,int64,0,2
Overall survival time,float64,0,466
...,...,...,...
D95%,float64,0,885
D97%,float64,0,844
D98%,float64,0,838
D99%,float64,0,804


ORN incidence: 198/1129 = 17.5%
Grade counts: {0: 931, 1: 36, 2: 39, 3: 54, 4: 69}


,Tumor site group,count,sum,mean,prevalence_pct
1,Major Salivary Glands,1,1,1.000000,100.000000
3,Oral Cavity,196,49,0.250000,25.000000
4,Oropharynx,740,138,0.186486,18.648649
5,Other,6,1,0.166667,16.666667
2,"Nasopharynx, Nasal Cavity, Paranasal Sinuses",23,2,0.086957,8.695652
0,Larynx/Hypopharynx,163,7,0.042945,4.294479


## 2. Dose-volume statistics and effect sizes

In [3]:
D_COLS=sorted([c for c in raw.columns if c.startswith('D') and c.endswith('%')],key=lambda c:float(c[1:-1]))
V_COLS=sorted([c for c in raw.columns if c.startswith('V') and c.endswith('Gy')],key=lambda c:float(c[1:-2]))
D_VOL=np.array([float(c[1:-1]) for c in D_COLS])
fig=go.Figure()
for k,label in [(0,'No ORN'),(1,'ORN')]:
    sub=raw.loc[y==k,D_COLS]
    fig.add_trace(go.Scatter(x=D_VOL,y=sub.mean().values,mode='lines',name=label))
fig.update_layout(title='Mean dose-at-volume curves (real cohort)',xaxis_title='Mandible volume (%)',yaxis_title='Dose (Gy)')
fig.show()
num=[c for c in ['Age','Smoking Pack-Years','D30%','D50%','D70%','V30Gy','V50Gy','V60Gy'] if c in raw.columns]
rows=[]
for c in num:
    a=raw.loc[y==0,c].dropna(); b=raw.loc[y==1,c].dropna()
    u,p=stats.mannwhitneyu(a,b)
    rows.append({'feature':c,'median_no_ORN':a.median(),'median_ORN':b.median(),
                 'median_difference':b.median()-a.median(),'MW_p':p,'AUC_univariate':roc_auc_score(y,raw[c]) if raw[c].notna().all() else np.nan})
stats_table=pd.DataFrame(rows).sort_values('MW_p')
display(stats_table)
if 'D30%' in raw:
    print(f"D30% medians: no ORN {raw.loc[y==0,'D30%'].median():.2f} Gy; ORN {raw.loc[y==1,'D30%'].median():.2f} Gy.")


,feature,median_no_ORN,median_ORN,median_difference,MW_p,AUC_univariate
2,D30%,47.77,57.475,9.705,1.320315e-24,0.731431
6,V50Gy,26.81,44.805,17.995,7.556243e-22,0.717088
7,V60Gy,11.83,23.875,12.045,2.227861e-21,0.714495
3,D50%,37.29,47.145,9.855,2.787572e-19,0.702894
5,V30Gy,68.14,79.395,11.255,1.115177e-14,0.674606
4,D70%,29.31,35.585,6.275,3.618674e-14,0.671188
0,Age,60.00,60.000,0.000,4.622300e-01,0.483389
1,Smoking Pack-Years,7.00,6.500,-0.500,8.030763e-01,0.494532


D30% medians: no ORN 47.77 Gy; ORN 57.48 Gy.


## 3. Phase 2 — DVH functional feature representation

In [4]:
dvh=raw[D_COLS].astype(float).values
dvh_scaler=StandardScaler()
dvh_z=dvh_scaler.fit_transform(dvh)
pca=PCA(n_components=min(10,dvh_z.shape[1]),random_state=SEED)
pcs=pca.fit_transform(dvh_z)
fig=make_subplots(rows=1,cols=2,subplot_titles=['Variance explained','First three DVH loadings'])
fig.add_trace(go.Bar(x=[f'PC{i+1}' for i in range(len(pca.explained_variance_ratio_))],
                     y=pca.explained_variance_ratio_*100,name='Individual'),1,1)
fig.add_trace(go.Scatter(x=[f'PC{i+1}' for i in range(len(pca.explained_variance_ratio_))],
                         y=np.cumsum(pca.explained_variance_ratio_)*100,mode='lines+markers',name='Cumulative'),1,1)
for i in range(min(3,len(pca.components_))):
    fig.add_trace(go.Scatter(x=D_VOL,y=pca.components_[i],name=f'PC{i+1} loading'),1,2)
fig.update_layout(height=430,title='DVH PCA (exploratory, real data)'); fig.show()
n95=int(np.searchsorted(np.cumsum(pca.explained_variance_ratio_),.95)+1)
print(f"{n95} components explain at least 95% of standardized D-curve variance.")
sil={}
for k in range(2,8):
    km=KMeans(n_clusters=k,n_init=10,random_state=SEED).fit(pcs[:,:min(6,pcs.shape[1])])
    from sklearn.metrics import silhouette_score
    sil[k]=silhouette_score(pcs[:,:min(6,pcs.shape[1])],km.labels_)
best_k=max(sil,key=sil.get)
km=KMeans(n_clusters=best_k,n_init=20,random_state=SEED).fit(pcs[:,:min(6,pcs.shape[1])])
cluster=pd.DataFrame({'cluster':km.labels_,'ORN':y,'D30':raw['D30%'] if 'D30%' in raw else raw[D_COLS[0]]})
ct=cluster.groupby('cluster').agg(n=('ORN','size'),ORN_cases=('ORN','sum'),ORN_rate=('ORN','mean'),mean_D30=('D30','mean')).reset_index()
display(ct)
px.bar(ct,x='cluster',y='ORN_rate',hover_data=['n','ORN_cases','mean_D30'],title='ORN rate by unsupervised DVH cluster').show()


3 components explain at least 95% of standardized D-curve variance.


,cluster,n,ORN_cases,ORN_rate,mean_D30
0,0,1019,195,0.191364,51.230128
1,1,110,3,0.027273,14.471545


## 4. Phase 3 — real-data ML baselines

In [5]:
exclude=[c for c in ['Study ID','ORN status','ORN Grade (Tsai)','Time to event','Overall survival','Overall survival time'] if c in raw.columns]
X=raw.drop(columns=exclude).copy()
X=pd.get_dummies(X,drop_first=False,dummy_na=True)
X=X.replace([np.inf,-np.inf],np.nan)
X=X.fillna(X.median(numeric_only=True)).fillna(0)
from sklearn.model_selection import train_test_split
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,random_state=SEED,stratify=y)
models={
 'Logistic regression':make_pipeline(StandardScaler(),LogisticRegression(max_iter=3000,C=.1)),
 'Random forest':RandomForestClassifier(n_estimators=300,min_samples_leaf=4,max_features='sqrt',class_weight='balanced',n_jobs=-1,random_state=SEED),
 'Extra Trees':ExtraTreesClassifier(n_estimators=300,min_samples_leaf=4,max_features='sqrt',class_weight='balanced',n_jobs=-1,random_state=SEED),
 'HistGradientBoosting':HistGradientBoostingClassifier(max_iter=200,learning_rate=.05,l2_regularization=2,random_state=SEED)
}
preds={}; rows=[]
for name,m in models.items():
    m.fit(Xtr,ytr); prob=m.predict_proba(Xte)[:,1]; pred=(prob>=.5).astype(int); preds[name]=prob
    rows.append({'Model':name,'ROC-AUC':roc_auc_score(yte,prob),'Average precision':average_precision_score(yte,prob),
                 'Brier':brier_score_loss(yte,prob),'F1@0.5':f1_score(yte,pred,zero_division=0),
                 'Recall@0.5':recall_score(yte,pred,zero_division=0),'Precision@0.5':precision_score(yte,pred,zero_division=0)})
results=pd.DataFrame(rows).sort_values('ROC-AUC',ascending=False)
display(results)
fig=go.Figure()
for name,p in preds.items():
    fpr,tpr,_=roc_curve(yte,p); fig.add_trace(go.Scatter(x=fpr,y=tpr,mode='lines',name=f'{name} AUC={roc_auc_score(yte,p):.3f}'))
fig.add_trace(go.Scatter(x=[0,1],y=[0,1],mode='lines',line=dict(dash='dash'),name='Chance'))
fig.update_layout(title='ROC curves (real ORN outcomes)',xaxis_title='False positive rate',yaxis_title='True positive rate');fig.show()
fig=go.Figure()
for name,p in preds.items():
    pr,rc,_=precision_recall_curve(yte,p);fig.add_trace(go.Scatter(x=rc,y=pr,mode='lines',name=f'{name} AP={average_precision_score(yte,p):.3f}'))
fig.add_hline(y=yte.mean(),line_dash='dash',annotation_text='Test prevalence')
fig.update_layout(title='Precision–recall curves',xaxis_title='Recall',yaxis_title='Precision');fig.show()
best=results.iloc[0]['Model']; print("Best ROC-AUC model:",best)
print(f"Holdout set n={len(yte)}; ORN prevalence={yte.mean():.1%}; best ROC-AUC={results.iloc[0]['ROC-AUC']:.3f}.")


,Model,ROC-AUC,Average precision,Brier,F1@0.5,Recall@0.5,Precision@0.5
0,Logistic regression,0.680914,0.278238,0.142855,0.040816,0.025,0.111111
2,Extra Trees,0.673387,0.298440,0.173149,0.340909,0.375,0.312500
1,Random forest,0.652151,0.296576,0.156951,0.238806,0.200,0.296296
3,HistGradientBoosting,0.634677,0.275790,0.159916,0.109091,0.075,0.200000


Best ROC-AUC model: Logistic regression
Holdout set n=226; ORN prevalence=17.7%; best ROC-AUC=0.681.


## 7. Findings generated from data


In [9]:
bestrow=results.iloc[0]
print("DATA-DERIVED FINDINGS")
print(f"• The MD Anderson table has {len(raw):,} patients and {int(y.sum()):,} ORN events ({y.mean():.1%}).")
print(f"• The best model on this single holdout split was {bestrow['Model']} (ROC-AUC {bestrow['ROC-AUC']:.3f}; AP {bestrow['Average precision']:.3f}).")
if 'D30%' in raw:
    print(f"• Median D30% was {raw.loc[y==0,'D30%'].median():.2f} Gy without ORN and {raw.loc[y==1,'D30%'].median():.2f} Gy with ORN.")
print("• The real CT archive is an independent cohort without confirmed ORN labels: image-derived metrics are not used to claim ORN prediction.")
print("• No clinical ORN CNN/LSTM/TFT result is reported because no real longitudinal CT sequences are paired with ORN outcomes.")


DATA-DERIVED FINDINGS
• The MD Anderson table has 1,129 patients and 198 ORN events (17.5%).
• The best model on this single holdout split was Logistic regression (ROC-AUC 0.681; AP 0.278).
• Median D30% was 47.77 Gy without ORN and 57.48 Gy with ORN.
• The real CT archive is an independent cohort without confirmed ORN labels: image-derived metrics are not used to claim ORN prediction.
• No clinical ORN CNN/LSTM/TFT result is reported because no real longitudinal CT sequences are paired with ORN outcomes.
